# RTM Demo (standalone, no messaging bus required)
* Small Demo that demonstrates the low latency benefits of RealTimeMode vs MicroBatchMode
  * MicroBatchMode is the defacto/default mode in structured Streaming
  * RealTimeMode is the new low latency mode in spark to allow for sub second streaming 
* In this demo we will do the following
  * Use Spark Rate Source to generate data
  * Apply Transformwithstate to do stateful operations on the data
  * Write to Spark Memory Sink with both MicroBatch mode (MBM) and RealTime Mode (RTM)
  * Calculate the latency differences between the 2

## Load resource files
* SensorDataGenerator - Uses spark nate rate source to generate records
* EnvironemntalMonitorListProcessor - TransformWithState Operator (Stateful Operation)
* HelperFunctions - Functions to simplify Demo

In [0]:
%run ./resources/SensorDataGenerator

In [0]:
%run ./resources/EnvironmentalMonitorListProcessor

In [0]:
%run ./resources/HelperFunctions

## Rate Source
* Use spark native rate source to generate records
* Set to generate 200 rows per second
* timestamp col is both the record generation timestamp and used for sensor timestamp
  * Effectively our "source timestamp"


In [ ]:
import com.databricks.dais2025.SensorDataGenerator

// note don't increase rps too high it will affect performance, this is setup to be a demonstration not benchmarking
// Default is 200 rowspersecond, can be changed via widget
val stream = SensorDataGenerator.createStream(spark, dbutils.widgets.get("rowsPerSecond").toInt, 2)

## Apply TransformFormWithState to Stream
* On the stream we apply [transformwithstate](https://docs.databricks.com/aws/en/stateful-applications/) operator to calculate the state of the senors per city and create alerts based on thresholds set in there
  * group by City
  * construct the output columns

In [0]:
import com.databricks.dais2025.tws.EnvironmentalMonitorListProcessor
import org.apache.spark.sql.streaming.{OutputMode, TimeMode}
import org.apache.spark.sql.functions.{col, struct, to_json, array, lit}
import com.databricks.dais2025.MyStructs._

val twsStream = stream
  .as[Input]
  .groupByKey(x => x.city)
  .transformWithState(
    new EnvironmentalMonitorListProcessor(), 
    TimeMode.ProcessingTime(), 
    OutputMode.Update()
  )
  .as[Output]
  .toDF()

## Shuffle Partitions
* Use 2 Shuffle partitions for shuffle stage (TWS)

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 2)

## WriteStream Options
* Checkpoint base path (widget) plus a per-run id; **MBM** and **RTM** each use a **separate** subdirectory (`.../mbm` and `.../rtm`) so the two queries do not share checkpoint state
* Memory sink table names for MBM and RTM runs
* Please update checkpointLocation (via widget) base path to a volume you have access to

In [0]:
val checkpointLocationRaw = dbutils.widgets.get("checkpointLocation")
val runId = s"rtmRunID${scala.util.Random.alphanumeric.take(6).mkString}"
val checkpointBase = if (checkpointLocationRaw.endsWith("/")) checkpointLocationRaw + runId else checkpointLocationRaw.stripSuffix("/") + "/" + runId
val checkpointLocationMbm = checkpointBase.stripSuffix("/") + "/mbm"
val checkpointLocationRtm = checkpointBase.stripSuffix("/") + "/rtm"

val tableName = s"rtm_mbm_demo_table${runId}"

### Run Stream in MBM Mode
* First do 20 batches of Mbm (Microbatch Mode) 
  * This is the defacto mode in spark structured streaming

In [0]:
import org.apache.spark.sql.streaming.Trigger

val queryProgressBatch = HelperFunctions.stopStreamAfterBatchesCollectProgress(
  spark = spark,
  stream = twsStream,
  queryName = "RTM-MBM-Demo",
  maxBatches = 20,
  trigger = Trigger.ProcessingTime("0 seconds"),
  checkpointLocation = checkpointLocationMbm,
  tableName = tableName
)

## MBM Latency Metrics
* MBM doesn't have Latency Metrics in the StreamingQueryProgress
* We calculate latency from the memory sink by subtracting source timestamp from sink timestamp
  * source timestamp = the timestamp from the rate source (when the record was generated)
  * sink timestamp = `current_timestamp()` captured when the record was written to the sink

In [0]:
HelperFunctions.printRTMStreamMetrics(queryProgressBatch)

In [0]:
import org.apache.spark.sql.functions.{avg, expr, percentile_approx, lit, dense_rank, round}
import org.apache.spark.sql.expressions.Window

val mbmLatencyDf = HelperFunctions.calculateLatencyFromMemoryTable(spark, tableName)
  .withColumn("batchnumberfortriggertype", dense_rank().over(Window.partitionBy("triggertype").orderBy("batch_id")))

val mbmLatencyNumbers = mbmLatencyDf
  .groupBy("triggertype", "batch_id", "batchnumberfortriggertype")
  .agg(
    avg("latency").alias("mean_latency"),
    percentile_approx(expr("latency"), lit(0.5), lit(100000)).alias("p50_latency"),
    percentile_approx(expr("latency"), lit(0.95), lit(100000)).alias("p95_latency"),
    percentile_approx(expr("latency"), lit(0.99), lit(100000)).alias("p99_latency")
  )
  .orderBy("batch_id")

display(mbmLatencyNumbers)

## Now Run In RealTime Mode
* We set trigger interval to 60 seconds (Default is 5 minutes)
  * This means that we checkpoint every 60 seconds, we only attempt to checkpoint then
  * If we used the default of 5 minutes the p99 would look better as well since less time is spent checkpointing
* We run 5 batches here, roughly a total of 5 minutes

In [0]:
import org.apache.spark.sql.streaming.Trigger

val queryProgressRTM = HelperFunctions.stopStreamAfterBatchesCollectProgress(
  spark = spark,
  stream = twsStream,
  queryName = "RTM-MBM-Demo",
  maxBatches = 5,
  trigger = Trigger.RealTime("60 seconds"),
  checkpointLocation = checkpointLocationRtm,
  tableName = tableName
)

## RTM Latency Metrics
* Real Time Mode has latency metrics directly in the StreamingQueryProgress, most importantly e2eLatency which gives us how long a record took from source to get to sink
* We extract per-batch p50/p95/p99 from the e2eLatency field instead of querying the memory sink table
* [Latency Metrics Info](https://docs.databricks.com/aws/en/structured-streaming/real-time#use-streamingqueryprogress)
* Note, there is generally some overhead on the first initial batch of realtime mode(and mbm) hence why we run multiple benches for demoing to show consistent low latency

In [0]:
HelperFunctions.printRTMStreamMetrics(queryProgressRTM)
// discuss numbers below

In [0]:
import org.apache.spark.sql.types._

val rtmLatencyRows = queryProgressRTM.zipWithIndex.flatMap { case (progress, idx) =>
  val (latenciesOpt, _) = HelperFunctions.parseProgressMetrics(progress)
  latenciesOpt.flatMap(_.e2eLatencyMs).map { e2e =>
    org.apache.spark.sql.Row("realtime", progress.batchId, (idx + 1).toLong, e2e.P50, e2e.P95, e2e.P99)
  }
}

val rtmLatencySchema = new StructType()
  .add("triggertype", StringType)
  .add("batch_id", LongType)
  .add("batchnumberfortriggertype", LongType)
  .add("p50_latency", LongType)
  .add("p95_latency", LongType)
  .add("p99_latency", LongType)

val rtmLatencyNumbers = spark.createDataFrame(
  spark.sparkContext.parallelize(rtmLatencyRows),
  rtmLatencySchema
).orderBy("batch_id")

display(rtmLatencyNumbers)

## MBM vs RTM Latency Comparison
* Compare avg p99 latency between MBM (from memory sink timestamp subtraction) and RTM (from e2eLatency in StreamingQueryProgress)
* Ignore first 2 batches for both modes as they have skewed startup times

In [0]:
val mbmP99 = mbmLatencyNumbers.select("triggertype", "batchnumberfortriggertype", "p99_latency")
val rtmP99 = rtmLatencyNumbers.select("triggertype", "batchnumberfortriggertype", "p99_latency")

display(
  mbmP99.union(rtmP99)
    .where("batchnumberfortriggertype not in (1,2)")
    .groupBy("triggertype")
    .agg(
      round(avg("p99_latency"), 2).alias("p99_latency")
    )
)

## Cleanup

In [0]:
// Clean up checkpoint dirs for this run (mbm + rtm under checkpointBase)
dbutils.fs.rm(checkpointBase, true)